# BRCA Subtype Classification
This notebook covers:
1. **EDA** — explore the raw data
2. **Model Results** — load results saved by each `src/train_*.py` script and display outputs

> **Before running sections 2+**, train models from terminal:
> ```bash
> cd src
> python train_random_forest.py
> python train_svm.py
> # python train_<any_new_model>.py
> ```

In [ ]:
import sys
import pickle
import pandas as pd
import altair as alt
from sklearn.metrics import ConfusionMatrixDisplay

sys.path.append('src')
from data_preprocessing import load_data, preprocess

alt.renderers.enable()

---
# 1. Exploratory Data Analysis

## Data Loading and Preview

In [ ]:
data, clinic = load_data()

Preview of genetic sequence data

In [ ]:
data.head()

Preview of Clinic data

In [ ]:
clinic.head()

## Counts of Samples per Subtype (raw clinic data)

In [ ]:
clinic_summary = clinic.groupby("Subtype").size().rename("Count").reset_index()
clinic_summary

In [ ]:
alt.Chart(clinic_summary).mark_bar().encode(
    alt.X("Subtype"),
    alt.Y("Count"),
    color="Subtype",
    tooltip=["Subtype", "Count"]
).properties(
    title="Number of Samples per Subtype",
    width=150,
    height=300
)

## Preprocessing

In [ ]:
df = preprocess(data, clinic)

Subtype sample counts of the final cleaned dataset

In [ ]:
df.groupby("Subtype").size().reset_index().rename(columns={0: "Count"})

## Gene Expression Scatter (two example genes)

In [ ]:
to_plot = df[["Subtype", "ENSG00000242268.2", "ENSG00000270112.3"]]
to_plot.columns = ['Subtype', 'gene_one', 'gene_two']

alt.Chart(to_plot).mark_circle().encode(
    alt.X("gene_one", scale=alt.Scale(zero=False)),
    alt.Y("gene_two", scale=alt.Scale(zero=False)),
    color="Subtype",
    size="gene_one"
).facet(
    facet="Subtype",
    columns=2
)

---
# 2. Model Results
Each section below loads the `.pkl` file saved by the corresponding `src/train_*.py` script.\
To add a new model: train it with a new script, save its results dict to `models/`, then add a section below.

---
## 2.1 Random Forest

In [ ]:
with open("models/rf_results.pkl", "rb") as f:
    rf = pickle.load(f)

print(f"Random Forest Accuracy: {rf['accuracy']:.4f}")

### Accuracy vs Max Depth

In [ ]:
max_depth_chart = alt.Chart(rf["max_depth_df"]).mark_line().encode(
    alt.X("max_depth", scale=alt.Scale(zero=False)),
    alt.Y("Accuracy", scale=alt.Scale(zero=False)),
    tooltip=["max_depth", "Accuracy"]
).properties(title="Random Forest Accuracy vs Max Depth")

max_depth_chart

### Accuracy vs n_estimators

In [ ]:
estimate_chart = alt.Chart(rf["n_estimate_df"]).mark_line().encode(
    alt.X("n_estimator", scale=alt.Scale(zero=False)),
    alt.Y("Accuracy", scale=alt.Scale(zero=False)),
    tooltip=["n_estimator", "Accuracy"]
).properties(title="Random Forest Accuracy vs n_estimator")

max_depth_chart | estimate_chart

### Confusion Matrix

In [ ]:
ConfusionMatrixDisplay(rf["conf_mat"], display_labels=rf["classes"]).plot()

**Random Forest Observations**\
The model is having trouble with predicting BRCA_Normal. This makes sense given that we have very few samples of this subtype.\
Also, the model is having problems with separating between BRCA_LumA and BRCA_LumB.

### Feature Importance (Top 10 Genes)

In [ ]:
rf["feature_data"]

In [ ]:
alt.Chart(rf["feature_data"]).mark_bar().encode(
    y=alt.Y("Feature Names", sort=alt.EncodingSortField(field="Importance", op="count", order="ascending"), title="Gene"),
    x=alt.X("Importance:Q"),
    color=alt.Color("Feature Names", title="Gene", legend=None),
    tooltip=["Feature Names", "Importance"]
)

---
## 2.2 Support Vector Machine (SVM)

In [ ]:
with open("models/svm_results.pkl", "rb") as f:
    svm = pickle.load(f)

print(f"SVM Accuracy: {svm['accuracy']:.4f}")

### Accuracy vs C Parameter

In [ ]:
alt.Chart(svm["svm_class_df"]).mark_line().encode(
    alt.X("C parameter", scale=alt.Scale(zero=False)),
    alt.Y("Accuracy", scale=alt.Scale(zero=False))
).properties(title="SVM Accuracy vs C Parameter")

### Confusion Matrix

In [ ]:
ConfusionMatrixDisplay(svm["conf_mat"], display_labels=svm["classes"]).plot()

**SVM Observation**\
The matrix shows that this model is not better than Random Forest in any subclassification.

---
## 2.3 Add New Model Here
1. Create `src/train_<model_name>.py` — train, evaluate, and save a results dict to `models/<model_name>_results.pkl`
2. Copy the block below and fill in your model's keys:

```python
with open("models/<model_name>_results.pkl", "rb") as f:
    my_model = pickle.load(f)

print(f"<Model> Accuracy: {my_model['accuracy']:.4f}")
ConfusionMatrixDisplay(my_model["conf_mat"], display_labels=my_model["classes"]).plot()
```